# ATP & WTA Tennis Match Predictor

Predicting singles match winners on **both** the men's (ATP) and women's (WTA) tours, using Elo ratings (overall + surface-specific) and XGBoost, trained on [Jeff Sackmann's data](https://github.com/JeffSackmann) (CC BY-NC-SA 4.0, 1991–present). Each tour is a separate model (Elo is only comparable within a tour). Pipeline per tour: **Elo → leakage-safe features → baselines vs XGBoost → evaluation**, then an **ATP-vs-WTA comparison** and on-demand prediction.

## 1. Setup

In [ ]:
# Auto-reload edited source modules so the kernel never uses a stale cached
# copy (run this Setup cell once per session; restart the kernel if you ever
# see "cannot import name ... from src.predict").
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score
from xgboost import XGBClassifier

from src.data import load_matches, clean_matches
from src.elo import compute_elo
from src.features import build_features, add_history_features, FEATURE_COLUMNS
from src.predict import save_artifacts, load_predictors, predict_match

XGB_PARAMS = dict(n_estimators=400, max_depth=4, learning_rate=0.05,
                  subsample=0.9, colsample_bytree=0.9, eval_metric="logloss")


## 2. Per-tour pipeline

`run_tour` runs the full leakage-safe pipeline for one tour: Elo → history features → time-based split → rank/Elo baselines → XGBoost → metrics, then trains a final model on all data and saves it to `../models/<tour>/`.

In [2]:
def run_tour(tour, train_end="2018-12-31", valid_end="2021-12-31"):
    """Full pipeline for one tour. Trains, evaluates, saves artifacts to
    ../models/<tour>/, and returns (metrics, ratings, names)."""
    df = clean_matches(load_matches("../data/raw", tour=tour))
    df_elo, ratings, surface_ratings = compute_elo(df)
    df_hist = add_history_features(df_elo, window=10)
    feats = build_features(df_hist, seed=42).dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)

    tr = feats[feats["tourney_date"] <= train_end]
    va = feats[(feats["tourney_date"] > train_end) & (feats["tourney_date"] <= valid_end)]
    te = feats[feats["tourney_date"] > valid_end]

    rank_acc = accuracy_score(te["a_won"], (te["rank_diff"] < 0).astype(int))
    elo_p = 1 / (1 + 10 ** (-te["elo_diff"].values / 400))
    elo_acc = accuracy_score(te["a_won"], (elo_p > 0.5).astype(int))

    model = XGBClassifier(**XGB_PARAMS)
    model.fit(tr[FEATURE_COLUMNS], tr["a_won"],
              eval_set=[(va[FEATURE_COLUMNS], va["a_won"])], verbose=False)
    xgb_p = model.predict_proba(te[FEATURE_COLUMNS])[:, 1]

    metrics = {
        "tour": tour.upper(), "matches": len(feats),
        "rank_acc": rank_acc, "elo_acc": elo_acc,
        "xgb_acc": accuracy_score(te["a_won"], (xgb_p > 0.5).astype(int)),
        "xgb_logloss": log_loss(te["a_won"], xgb_p),
        "xgb_brier": brier_score_loss(te["a_won"], xgb_p),
        "xgb_auc": roc_auc_score(te["a_won"], xgb_p),
    }

    final = XGBClassifier(**XGB_PARAMS)
    final.fit(feats[FEATURE_COLUMNS], feats["a_won"])
    names = (pd.concat([
        df[["winner_id", "winner_name"]].rename(columns={"winner_id": "id", "winner_name": "name"}),
        df[["loser_id", "loser_name"]].rename(columns={"loser_id": "id", "loser_name": "name"}),
    ]).drop_duplicates("id").set_index("id")["name"].to_dict())
    state = {"ratings": ratings, "surface_ratings": surface_ratings,
             "names": names, "feature_columns": FEATURE_COLUMNS}
    save_artifacts(f"../models/{tour}", final, state)
    return metrics, ratings, names


In [3]:
atp_metrics, atp_ratings, atp_names = run_tour("atp")
wta_metrics, wta_ratings, wta_names = run_tour("wta")
print("done: atp", atp_metrics["matches"], "matches | wta", wta_metrics["matches"], "matches")


done: atp 105389 matches | wta 72823 matches


## 3. ATP vs WTA — model comparison (held-out test set, 2022–present)

In [4]:
comp = pd.DataFrame([atp_metrics, wta_metrics]).set_index("tour")
display(comp.round(4))


,matches,rank_acc,elo_acc,xgb_acc,xgb_logloss,xgb_brier,xgb_auc
tour,,,,,,,
ATP,105389,0.6392,0.6413,0.6533,0.6160,0.2143,0.7164
WTA,72823,0.6249,0.6495,0.6545,0.6186,0.2154,0.7129


## 4. Top players by Elo, each tour

In [5]:
def top_elo(ratings, names, n=10):
    top = sorted(ratings.items(), key=lambda kv: -kv[1])[:n]
    return pd.DataFrame([(names.get(i, i), round(v)) for i, v in top],
                        columns=["player", "elo"])

print("Top 10 ATP by Elo"); display(top_elo(atp_ratings, atp_names))
print("Top 10 WTA by Elo"); display(top_elo(wta_ratings, wta_names))


Top 10 ATP by Elo


,player,elo
0,Jannik Sinner,2338
1,Carlos Alcaraz,2222
2,Roger Federer,2131
3,Robin Soderling,2105
4,Novak Djokovic,2088
5,Alexander Zverev,2058
6,Juan Martin del Potro,2022
7,Rafael Nadal,2022
8,Patrick Rafter,2018
9,Daniil Medvedev,1987


Top 10 WTA by Elo


,player,elo
0,Justine Henin,2310
1,Steffi Graf,2283
2,Aryna Sabalenka,2248
3,Ashleigh Barty,2204
4,Lindsay Davenport,2179
5,Elena Rybakina,2157
6,Jennifer Capriati,2130
7,Na Li,2129
8,Jessica Pegula,2101
9,Coco Gauff,2082


## 5. On-demand prediction

Pass `tour="atp"` or `tour="wta"`. `predict_match` echoes the resolved full names (fuzzy-matched and disambiguated by Elo within that tour) so you can confirm it picked the players you meant. Surfaces are case-insensitive; an unknown tour, surface, or name raises a clear error.

In [ ]:
# Self-contained: this cell works on its own (fresh kernel) as long as the
# models exist on disk under ../models/. If load_predictors errors with
# FileNotFoundError, run section 2's run_tour cell once to create them.
import sys; sys.path.append("..")
from src.predict import load_predictors, predict_match

preds = load_predictors("../models")

matchups = [
    ("Sinner", "Medvedev", "Hard", "atp"),
    ("Alcaraz", "Djokovic", "Clay", "atp"),
    ("Swiatek", "Sabalenka", "Clay", "wta"),
    ("Gauff", "Swiatek", "Hard", "wta"),
]
for a, b, surface, tour in matchups:
    r = predict_match(preds, a, b, surface, tour=tour)
    print(f"[{r['tour'].upper():3}] {r['player_a']:>18} vs {r['player_b']:<18} "
          f"on {r['surface']:<5}  P({r['player_a'].split()[-1]}) = {r['prob']:.1%}")


### How to use the predictor

`predict_match(preds, player_a, player_b, surface, tour)` returns the win probability for **`player_a`** plus the resolved player names.

```python
r = predict_match(preds, "Alcaraz", "Sinner", "Clay", tour="atp")
print(r["player_a"], "vs", r["player_b"], "->", f"{r['prob']:.1%}")
# Carlos Alcaraz vs Jannik Sinner -> 43.8%
```

**The arguments**
- `preds` — both tours' models, loaded once with `load_predictors("../models")`. (Run the per-tour training section above first so `../models/atp/` and `../models/wta/` exist.)
- `player_a`, `player_b` — names. Matching is **fuzzy and case-insensitive** and **disambiguated by Elo** within the chosen tour, so `"Sinner"` → Jannik Sinner and the typo `"Alcarez"` → Carlos Alcaraz. A name that isn't a real player in the dataset raises a clear error instead of silently picking a namesake.
- `surface` — `"Hard"`, `"Clay"`, or `"Grass"` (case-insensitive; an unknown surface raises).
- `tour` — **`"atp"`** (men) or **`"wta"`** (women). **Required** — each tour is its own model, and players are only looked up within that tour. An unknown tour raises with the list of available tours.

**The result** is a dict `{prob, player_a, player_b, surface, tour}`; `prob` is `player_a`'s win probability (0–1). Use `predict_proba(preds, a, b, surface, tour)` if you just want the number.

**Try your own matchup** — copy into a new cell and edit the four values:

```python
a, b, surface, tour = "Swiatek", "Gauff", "Hard", "wta"
r = predict_match(preds, a, b, surface, tour=tour)
print(f"[{r['tour'].upper()}] P({r['player_a']} beats {r['player_b']} on {r['surface']}) = {r['prob']:.1%}")
```

**Good to know**
- **Tour-level only.** The data is ATP/WTA *main-draw*, so lower-tier players (ITF / futures / qualifying) aren't included — naming one raises a "not in the dataset" error rather than returning a fake number.
- **Elo-driven for hypothetical matches.** Recent-form, head-to-head, and rest features are set neutral when scoring a match that hasn't happened, so on-demand probabilities lean on Elo and are a bit more extreme than the model's measured ~0.65 test-set accuracy.